# 訓練後・デプロイ段階の防御（Post-training / Deployment Stage Defense）

訓練後段階では、潜在的にバックドア化された訓練済みモデルと（オプションで）少量のクリーンデータを用いて、バックドアの検出や除去を行う。デプロイ段階では、量子化モデルへの重み攻撃に対する耐性強化や検証を行う。

## バックドア検出（Backdoor Detection）

モデルにバックドアが存在するかどうかを判定する。

### 特徴ベースの検出

| 手法 | アプローチ |
|------|----------|
| **ABS** | 潜在的に侵害されたニューロンを特定し、トリガーの逆工学で確認 |
| **EX-RAY** | 対称的な特徴差分で検出 |
| **DECREE** | 事前学習済みエンコーダ向けの検出手法。最小トリガーパターンを探索 |
| **DeepInspect** | ブラックボックス設定でモデル反転を通じて代替訓練データを復元して検出 |

### 重みベースの検出

| 手法 | アプローチ |
|------|----------|
| **ULPs** | 多数のクリーン/毒入れモデルを訓練し、Universal Litmus Patterns（普遍的リトマス試験紙）で検出 |
| **MNTD** | Shadow modelを用いた検出 |

## 目標ラベルの予測（Target Label Prediction）

バックドアが検出された場合に、攻撃者が狙った目標ラベルを特定する。

### 特徴ベース

| 手法 | アプローチ |
|------|----------|
| **NC（Neural Cleanse）** | 各ラベルに対する最小の普遍的摂動（マスク化トリガー）を逆算。異常に小さいトリガーが見つかるラベルが目標ラベル |
| **TND** | データが限られた/ない状況にも対応 |
| **K-Arm** | 強化学習のK-arm bandit戦略で効率的に目標ラベルを特定 |
| **L-RED** | ラグランジュベースの逆工学 |
| **B3D** | 勾配なし最適化による逆工学 |

### 重みベース

| 手法 | アプローチ |
|------|----------|
| **Greg et al.** | 最終線形層でトロイの目標ラベルを関連付け |
| **CPBD** | 臨界経路分析で特定 |

## バックドアの除去（Backdoor Removal）

バックドアを検出した後、モデルの有用性を保ちつつバックドアを除去する。

### 構造修正（プルーニング）ベース

バックドアに関連するニューロンを特定して除去する。

| 手法 | アプローチ |
|------|----------|
| **FP（Fine-Pruning）** | 良性データで活性化できないニューロンをプルーニング後にファインチューニング |
| **ANP** | min-max最適化で危険なニューロンを特定して除去 |
| **ShapPruning** | シャプレイ値で各ニューロンの貢献度を計算し除去 |
| **CLP** | チャネルのリプシッツ定数を測定し、異常に大きいチャネルを除去 |
| **AWM** | ソフトな重みマスキング |
| **NPD** | 線形変換層を挿入して無力化 |

### ファインチューニングベース

トリガーの逆工学と組み合わせてモデルを再学習する。

$$
\min_{\boldsymbol{\theta}, m, \Delta} \sum_{(\mathbf{x}, y) \in \mathcal{D}_{\text{clean}}} \mathcal{L}\left(f_{\boldsymbol{\theta}}((1-m) \odot \mathbf{x} + m \odot \Delta), y\right)
$$

- $m$: トリガーのマスク
- $\Delta$: 逆転されたトリガーパターン

| 手法 | アプローチ |
|------|----------|
| **NC + i-BAU** | 逆転トリガーを用いた再学習 |
| **MESA** | Max-entropy staircase approximatorでトリガー分布を近似 |
| **PBE** | 敵対的サンプルを利用した再学習 |
| **NAD** | 教師モデルのファインチューニングによる知識蒸留ベースの除去 |

## 重み攻撃への防御（Weight Attack Defense）

重み攻撃（ビットフリップ攻撃）は、デプロイされたモデルの量子化された重みパラメータを直接改変する攻撃である。

### モデル強化

| 手法 | アプローチ |
|------|----------|
| **OCM** | 部分的に重複するビット文字列を採用し、tanh活性化関数で重みを相関化 |
| **Aegis** | 中間層に内部分類器（ICs）を追加し、動的終了機構を実装 |
| **RREC** | ランダム回転でビット順を難読化し、非線形量子化で圧縮 |

### フィンガープリント検証

| 手法 | アプローチ |
|------|----------|
| **DeepAttest** | デバイス固有のフィンガープリントをモデル重みに符号化し、信頼実行環境（TEE）に保存。推論時に検証 |

## 参考文献

- [Wu et al. (2023). Defenses in Adversarial Machine Learning: A Survey. Section V.](https://arxiv.org/abs/2312.08890)
- [Wang et al. (2019). Neural Cleanse: Identifying and Mitigating Backdoor Attacks in Neural Networks.](https://ieeexplore.ieee.org/document/8835365)
- [Liu et al. (2018). Fine-Pruning: Defending Against Backdooring Attacks on Deep Neural Networks.](https://arxiv.org/abs/1805.12185)
- [Kolouri et al. (2020). Universal Litmus Patterns: Revealing Backdoor Attacks in CNNs.](https://arxiv.org/abs/1906.10842)